## RWF-2000 + UCF-Crime Fighting + Bus Violence
### TEST ONLY — tanpa training

Notebook ini mengevaluasi **invariansi FrameMean terhadap urutan temporal**
pada tiga held-out dataset:

1. RWF-2000
2. UCF-Crime Fighting
3. Bus Violence

Tidak ada pooling antar-held-out dataset.

Skenario:
- Original
- Reverse
- 10 × Random Shuffle

FrameMean menggunakan:

```python
context = features.mean(dim=1)
```

sehingga secara matematis permutation-invariant.

Untuk seed berikutnya, setelah path dataset benar, cukup ubah:

```python
RUN_SEED = 0
```

menjadi `42` atau `123`.

In [1]:
!pip -q install timm

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import json
import random
import platform
import warnings
import zlib
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from timm import create_model
from tqdm.auto import tqdm
from IPython.display import display

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_fscore_support,
)

warnings.filterwarnings("ignore", category=UserWarning)

print("Python :", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))


Python : 3.13.15
PyTorch: 2.11.0+cu128
CUDA   : True
GPU    : NVIDIA A100-SXM4-40GB


## 1. Konfigurasi

In [ ]:
# ============================================================
# SATU-SATUNYA VARIABEL YANG DIUBAH ANTAR-SEED
# ============================================================
RUN_SEED = 0

VALID_SEEDS = (
    0,
    42,
    123,
)

if RUN_SEED not in VALID_SEEDS:
    raise ValueError(
        f"RUN_SEED harus salah satu dari {VALID_SEEDS}; "
        f"mendapat {RUN_SEED}."
    )

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

# ============================================================
# EXACT FRAMEMEAN PROTOCOL
# ============================================================
VARIANT = "frame_mean"

SEQUENCE_LENGTH = 16
RAW_HEIGHT = 64
RAW_WIDTH = 64
BACKBONE_INPUT_SIZE = 224
NUM_CLASSES = 2

NUM_RANDOM_SHUFFLES = 10
PERMUTATION_BASE_SEED = 20260810

BATCH_SIZE = 8
NUM_WORKERS = 2

PREFER_LEGACY_CACHE = True
LEGACY_CACHE_COLOR_ORDER = "BGR"
AUTO_SCALE_LEGACY_CACHE = True
ALLOW_BUILD_CACHE = True

PROB_ATOL = 1e-6
METRIC_ATOL = 1e-12
AUC_ATOL = 1e-12
AP_ATOL = 1e-12

# ============================================================
# CHECKPOINT
# ============================================================
MODEL_ROOT = Path(
    "/content/drive/MyDrive/Models"
)

CHECKPOINT_PATH_OVERRIDE = None


def resolve_checkpoint(
    seed: int,
) -> Path:
    if CHECKPOINT_PATH_OVERRIDE is not None:
        path = Path(
            CHECKPOINT_PATH_OVERRIDE
        )

        if not path.exists():
            raise FileNotFoundError(
                f"CHECKPOINT_PATH_OVERRIDE tidak ditemukan: {path}"
            )

        return path

    candidates = [
        (
            MODEL_ROOT
            / f"EffTempNet_FrameMean_NoLS_{seed}Seeds_v1"
            / VARIANT
            / f"seed_{seed}"
            / "best.pt"
        ),
        (
            MODEL_ROOT
            / f"EffTempNet_FrameMean_NoLS_Seed{seed}_v1"
            / VARIANT
            / f"seed_{seed}"
            / "best.pt"
        ),
        (
            MODEL_ROOT
            / f"EffTempNet_FrameMean_NoLS_{seed}Seed_v1"
            / VARIANT
            / f"seed_{seed}"
            / "best.pt"
        ),
    ]

    matches = [
        path
        for path in candidates
        if path.exists()
    ]

    matches = list(
        dict.fromkeys(
            matches
        )
    )

    if len(
        matches
    ) == 1:
        return matches[
            0
        ]

    if len(
        matches
    ) > 1:
        raise RuntimeError(
            "Ditemukan beberapa checkpoint FrameMean. "
            "Set CHECKPOINT_PATH_OVERRIDE:\n"
            + "\n".join(
                str(
                    path
                )
                for path
                in matches
            )
        )

    recursive = sorted(
        MODEL_ROOT.glob(
            f"**/{VARIANT}/seed_{seed}/best.pt"
        )
    )

    recursive = [
        path
        for path in recursive
        if (
            "framemean"
            in str(
                path
            )
            .lower()
            .replace(
                "_",
                "",
            )
        )
    ]

    if len(
        recursive
    ) == 1:
        return recursive[
            0
        ]

    if len(
        recursive
    ) == 0:
        raise FileNotFoundError(
            f"best.pt FrameMean seed {seed} tidak ditemukan di {MODEL_ROOT}.\n"
            "Jika naming folder berbeda, isi CHECKPOINT_PATH_OVERRIDE."
        )

    raise RuntimeError(
        "Ditemukan beberapa checkpoint FrameMean yang ambigu. "
        "Set CHECKPOINT_PATH_OVERRIDE:\n"
        + "\n".join(
            str(
                path
            )
            for path
            in recursive
        )
    )


CHECKPOINT_PATH = resolve_checkpoint(
    RUN_SEED
)

# ============================================================
# HELD-OUT DATA
# ============================================================
BENCHMARK_ROOT = Path(
    "/content/drive/MyDrive/Benchmark"
)

EXISTING_CACHE_DIR = (
    MODEL_ROOT
    / "new_saved_dataset"
)

HELDOUT_CACHE_DIR = (
    MODEL_ROOT
    / "heldout_cache_rgb64_v2"
)

CLASS_NAMES = [
    "NonViolence",
    "Violence",
]

NONVIOLENCE_ALIASES = (
    "NonViolence",
    "Non-Violence",
    "Non_Violence",
    "Nonviolent",
    "NonViolent",
    "NonFight",
    "Non-Fight",
    "Non_Fight",
    "Normal",
    "NormalVideos",
    "Normal_Videos",
)

VIOLENCE_ALIASES = (
    "Violence",
    "Violent",
    "Fight",
    "Fighting",
    "FightVideos",
    "ViolentVideos",
    "Abnormal",
)

HELDOUT_SPECS = {
    "RWF": {
        "raw_dir": (
            BENCHMARK_ROOT
            / "RWF"
        ),
        "cache_name": "RWF",
        "cache_dir": EXISTING_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },

    "UCF": {
        "raw_dir": (
            BENCHMARK_ROOT
            / "UCF"
        ),
        "cache_name": "UCF",
        "cache_dir": HELDOUT_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },

    "BV": {
        "raw_dir": (
            BENCHMARK_ROOT
            / "BV"
        ),
        "cache_name": "BV",
        "cache_dir": HELDOUT_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },
}

OUTPUT_DIR = (
    MODEL_ROOT
    / "HeldOut_Evaluation"
    / "FrameMean_TemporalOrder"
    / f"seed_{RUN_SEED}"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

HELDOUT_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    "Checkpoint :",
    CHECKPOINT_PATH,
)

print(
    "Exists     :",
    CHECKPOINT_PATH.exists(),
)

print(
    "Output     :",
    OUTPUT_DIR,
)

print(
    "Device     :",
    DEVICE,
)

print(
    "Shuffles   :",
    NUM_RANDOM_SHUFFLES,
)

print(
    "Perm seed  :",
    PERMUTATION_BASE_SEED,
)

Checkpoint : /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/EffTempNet_FrameMean_NoLS_0Seeds_v1/frame_mean/seed_0/best.pt
Exists     : True
Output     : /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/FrameMean_TemporalOrder/seed_0
Device     : cuda
Shuffles   : 10
Perm seed  : 20260810


## 2. Cache pengujian

In [5]:
def set_seed(
    seed,
):
    os.environ[
        "PYTHONHASHSEED"
    ] = str(
        seed
    )

    os.environ.setdefault(
        "CUBLAS_WORKSPACE_CONFIG",
        ":4096:8",
    )

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    if (
        hasattr(
            torch.backends,
            "cuda",
        )
        and hasattr(
            torch.backends.cuda,
            "matmul",
        )
    ):
        torch.backends.cuda.matmul.allow_tf32 = False

    if hasattr(
        torch.backends,
        "cudnn",
    ):
        torch.backends.cudnn.allow_tf32 = False

    try:
        torch.set_float32_matmul_precision(
            "highest"
        )
    except Exception:
        pass

    try:
        torch.use_deterministic_algorithms(
            True,
            warn_only=True,
        )
    except TypeError:
        try:
            torch.use_deterministic_algorithms(
                True
            )
        except Exception:
            pass


def seed_worker(
    worker_id,
):
    worker_seed = (
        torch.initial_seed()
        % (2**32)
    )

    np.random.seed(
        worker_seed
    )

    random.seed(
        worker_seed
    )


def safe_torch_load(
    path,
    map_location,
):
    try:
        return torch.load(
            path,
            map_location=map_location,
            weights_only=False,
        )

    except TypeError:
        return torch.load(
            path,
            map_location=map_location,
        )


set_seed(
    RUN_SEED
)

# ============================================================
# SELF-CONTAINED CACHE BUILDER
# ============================================================
# Function default arguments are evaluated at function-definition time.
# Therefore the cache builder does not depend on SEQUENCE_LENGTH / RAW_HEIGHT /
# RAW_WIDTH being already defined in another cell.
_CACHE_BUILDER_SEQUENCE_LENGTH = 16
_CACHE_BUILDER_RAW_HEIGHT = 64
_CACHE_BUILDER_RAW_WIDTH = 64

if (
    "SEQUENCE_LENGTH" in globals()
    and int(
        globals()[
            "SEQUENCE_LENGTH"
        ]
    )
    != _CACHE_BUILDER_SEQUENCE_LENGTH
):
    raise RuntimeError(
        "Protocol mismatch: SEQUENCE_LENGTH harus 16."
    )

if (
    "RAW_HEIGHT" in globals()
    and int(
        globals()[
            "RAW_HEIGHT"
        ]
    )
    != _CACHE_BUILDER_RAW_HEIGHT
):
    raise RuntimeError(
        "Protocol mismatch: RAW_HEIGHT harus 64."
    )

if (
    "RAW_WIDTH" in globals()
    and int(
        globals()[
            "RAW_WIDTH"
        ]
    )
    != _CACHE_BUILDER_RAW_WIDTH
):
    raise RuntimeError(
        "Protocol mismatch: RAW_WIDTH harus 64."
    )

VIDEO_EXTENSIONS = {
    ".mp4", ".avi", ".mov", ".mkv",
    ".mpeg", ".mpg", ".webm"
}

def direct_subdirs(root: Path) -> Dict[str, Path]:
    if not root.exists():
        return {}
    return {
        p.name: p
        for p in root.iterdir()
        if p.is_dir()
    }

def resolve_one_class_dir(
    root: Path,
    explicit_name: Optional[str],
    aliases: Tuple[str, ...],
    canonical_name: str,
) -> Path:
    if explicit_name is not None:
        candidate = root / explicit_name
        if not candidate.exists():
            raise FileNotFoundError(
                f"Folder {canonical_name} tidak ditemukan: {candidate}"
            )
        return candidate

    subdirs = direct_subdirs(root)
    lower_map = {
        name.lower(): path
        for name, path in subdirs.items()
    }

    matches = []
    for alias in aliases:
        candidate = lower_map.get(alias.lower())
        if candidate is not None and candidate not in matches:
            matches.append(candidate)

    if len(matches) != 1:
        raise RuntimeError(
            f"Tidak dapat menentukan folder kelas {canonical_name} "
            f"secara aman pada {root}.\n"
            f"Matches: {[p.name for p in matches]}\n"
            f"Available: {sorted(subdirs.keys())}\n"
            "Isi nonviolence_dir/violence_dir di HELDOUT_SPECS."
        )

    return matches[0]

def list_videos_for_spec(
    raw_dir: Path,
    nonviolence_dir: Optional[str],
    violence_dir: Optional[str],
) -> List[Tuple[Path, int]]:
    if not raw_dir.exists():
        raise FileNotFoundError(
            f"Raw held-out directory tidak ditemukan: {raw_dir}"
        )

    class_dirs = {
        0: resolve_one_class_dir(
            raw_dir,
            nonviolence_dir,
            NONVIOLENCE_ALIASES,
            "NonViolence",
        ),
        1: resolve_one_class_dir(
            raw_dir,
            violence_dir,
            VIOLENCE_ALIASES,
            "Violence",
        ),
    }

    records = []

    for label, class_dir in class_dirs.items():
        videos = sorted(
            p for p in class_dir.rglob("*")
            if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
        )

        if not videos:
            raise RuntimeError(
                f"Tidak ada video pada folder kelas: {class_dir}"
            )

        print(
            f"{CLASS_NAMES[label]:12s}: {len(videos):5d} clips | {class_dir}"
        )

        records.extend(
            (p, label)
            for p in videos
        )

    labels = np.asarray(
        [label for _, label in records],
        dtype=np.int64,
    )

    if set(np.unique(labels).tolist()) != {0, 1}:
        raise RuntimeError(
            "Held-out dataset harus mempunyai kedua kelas: "
            "NonViolence dan Violence."
        )

    return records

def extract_uniform_rgb_frames(
    video_path: Path,
    sequence_length: int = _CACHE_BUILDER_SEQUENCE_LENGTH,
    height: int = _CACHE_BUILDER_RAW_HEIGHT,
    width: int = _CACHE_BUILDER_RAW_WIDTH,
) -> Optional[np.ndarray]:
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        return None

    frame_count = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    if frame_count <= 0:
        cap.release()
        return None

    frame_positions = np.linspace(
        0,
        max(frame_count - 1, 0),
        sequence_length,
    ).astype(int)

    frames = []

    for pos in frame_positions:
        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(pos),
        )

        ok, frame_bgr = cap.read()

        if not ok:
            break

        frame_rgb = cv2.cvtColor(
            frame_bgr,
            cv2.COLOR_BGR2RGB,
        )

        frame_rgb = cv2.resize(
            frame_rgb,
            (width, height),
            interpolation=cv2.INTER_LINEAR,
        )

        frames.append(
            frame_rgb.astype(np.float32) / 255.0
        )

    cap.release()

    if not frames:
        return None

    while len(frames) < sequence_length:
        frames.append(frames[-1].copy())

    return np.stack(
        frames[:sequence_length],
        axis=0,
    ).astype(np.float32)

def build_rgb_cache(
    raw_dir: Path,
    cache_dir: Path,
    cache_name: str,
    nonviolence_dir: Optional[str],
    violence_dir: Optional[str],
):
    cache_dir.mkdir(parents=True, exist_ok=True)

    feature_path = cache_dir / f"{cache_name}_rgb64_v2_features.npy"
    label_path = cache_dir / f"{cache_name}_rgb64_v2_labels.npy"
    paths_path = cache_dir / f"{cache_name}_rgb64_v2_paths.json"

    records = list_videos_for_spec(
        raw_dir,
        nonviolence_dir,
        violence_dir,
    )

    # Gunakan temporary memmap agar tidak menyimpan semua clip di RAM.
    n_requested = len(records)

    temp_path = cache_dir / f".{cache_name}_tmp_features.npy"

    if temp_path.exists():
        temp_path.unlink()

    temp = np.lib.format.open_memmap(
        temp_path,
        mode="w+",
        dtype=np.float32,
        shape=(
            n_requested,
            _CACHE_BUILDER_SEQUENCE_LENGTH,
            _CACHE_BUILDER_RAW_HEIGHT,
            _CACHE_BUILDER_RAW_WIDTH,
            3,
        ),
    )

    labels_buffer = np.empty(
        n_requested,
        dtype=np.int64,
    )

    valid_paths = []
    failed_paths = []
    valid_count = 0

    for video_path, label in tqdm(
        records,
        desc=f"Build cache {cache_name}",
        unit="video",
    ):
        clip = extract_uniform_rgb_frames(video_path)

        if clip is None:
            failed_paths.append(str(video_path))
            continue

        temp[valid_count] = clip
        labels_buffer[valid_count] = int(label)
        valid_paths.append(str(video_path))
        valid_count += 1

    temp.flush()
    del temp

    if valid_count == 0:
        if temp_path.exists():
            temp_path.unlink()
        raise RuntimeError(
            f"Semua video gagal didekode untuk {cache_name}."
        )

    # Compact hanya sample valid ke final .npy.
    temp_read = np.load(
        temp_path,
        mmap_mode="r",
    )

    final = np.lib.format.open_memmap(
        feature_path,
        mode="w+",
        dtype=np.float32,
        shape=(
            valid_count,
            _CACHE_BUILDER_SEQUENCE_LENGTH,
            _CACHE_BUILDER_RAW_HEIGHT,
            _CACHE_BUILDER_RAW_WIDTH,
            3,
        ),
    )

    chunk_size = 16

    for start in range(0, valid_count, chunk_size):
        end = min(start + chunk_size, valid_count)
        final[start:end] = temp_read[start:end]

    final.flush()
    del final, temp_read

    np.save(
        label_path,
        labels_buffer[:valid_count],
    )

    with open(
        paths_path,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            {
                "paths": valid_paths,
                "failed": failed_paths,
                "n_requested": n_requested,
                "n_valid": valid_count,
                "color_order": "RGB",
                "sequence_length": _CACHE_BUILDER_SEQUENCE_LENGTH,
                "height": _CACHE_BUILDER_RAW_HEIGHT,
                "width": _CACHE_BUILDER_RAW_WIDTH,
            },
            f,
            indent=2,
        )

    if temp_path.exists():
        temp_path.unlink()

    print(
        f"Saved {feature_path} | "
        f"shape=({valid_count},{_CACHE_BUILDER_SEQUENCE_LENGTH},{_CACHE_BUILDER_RAW_HEIGHT},{_CACHE_BUILDER_RAW_WIDTH},3)"
    )
    print("Failed videos:", len(failed_paths))

    return {
        "feature_path": feature_path,
        "label_path": label_path,
        "paths_path": paths_path,
        "color_order": "RGB",
        "cache_source": "built_rgb64_v2",
    }

def resolve_heldout_store(
    dataset_name: str,
    spec: Dict,
) -> Dict:
    raw_dir = Path(spec["raw_dir"])
    cache_dir = Path(spec["cache_dir"])
    cache_name = str(spec["cache_name"])

    rgb_feature = cache_dir / f"{cache_name}_rgb64_v2_features.npy"
    rgb_label = cache_dir / f"{cache_name}_rgb64_v2_labels.npy"
    rgb_paths = cache_dir / f"{cache_name}_rgb64_v2_paths.json"

    legacy_feature = cache_dir / f"{cache_name}_features.npy"
    legacy_label = cache_dir / f"{cache_name}_labels.npy"
    legacy_paths = cache_dir / f"{cache_name}_paths.json"

    # Sama dengan training: RGB-v2 diprioritaskan; legacy dipakai jika v2 tidak ada.
    if rgb_feature.exists() and rgb_label.exists():
        print(f"✅ {dataset_name}: RGB-v2 cache")
        return {
            "feature_path": rgb_feature,
            "label_path": rgb_label,
            "paths_path": rgb_paths if rgb_paths.exists() else None,
            "color_order": "RGB",
            "cache_source": "rgb64_v2",
        }

    if (
        PREFER_LEGACY_CACHE
        and legacy_feature.exists()
        and legacy_label.exists()
    ):
        print(
            f"✅ {dataset_name}: legacy cache "
            f"({LEGACY_CACHE_COLOR_ORDER}→RGB saat loading)"
        )
        return {
            "feature_path": legacy_feature,
            "label_path": legacy_label,
            "paths_path": legacy_paths if legacy_paths.exists() else None,
            "color_order": LEGACY_CACHE_COLOR_ORDER,
            "cache_source": "legacy",
        }

    if not ALLOW_BUILD_CACHE:
        raise FileNotFoundError(
            f"Cache {dataset_name} tidak ada dan ALLOW_BUILD_CACHE=False."
        )

    if not raw_dir.exists():
        raise FileNotFoundError(
            f"Cache dan raw folder tidak ditemukan untuk {dataset_name}.\n"
            f"Raw expected: {raw_dir}"
        )

    print(
        f"ℹ️ {dataset_name}: cache belum ada, dibuat dari {raw_dir}"
    )

    return build_rgb_cache(
        raw_dir=raw_dir,
        cache_dir=cache_dir,
        cache_name=cache_name,
        nonviolence_dir=spec.get("nonviolence_dir"),
        violence_dir=spec.get("violence_dir"),
    )

# ============================================================
# RESOLVE HELD-OUT STORES
# ============================================================
TEST_STORES = {}

dataset_audit_rows = []

for (
    dataset_name,
    spec,
) in HELDOUT_SPECS.items():
    print(
        "\n"
        + "=" * 90
    )

    print(
        "DATASET:",
        dataset_name,
    )

    print(
        "=" * 90
    )

    store = resolve_heldout_store(
        dataset_name,
        spec,
    )

    features = np.load(
        store[
            "feature_path"
        ],
        mmap_mode="r",
    )

    labels = np.load(
        store[
            "label_path"
        ],
        mmap_mode="r",
    )

    expected_shape = (
        len(
            labels
        ),
        16,
        64,
        64,
        3,
    )

    if features.shape != expected_shape:
        raise ValueError(
            f"{dataset_name}: {features.shape} != {expected_shape}"
        )

    if set(
        np.unique(
            labels
        ).tolist()
    ) != {
        0,
        1,
    }:
        raise ValueError(
            f"{dataset_name}: held-out dataset harus mempunyai dua kelas."
        )

    TEST_STORES[
        dataset_name
    ] = (
        store[
            "feature_path"
        ],
        store[
            "label_path"
        ],
        store[
            "color_order"
        ],
        store.get(
            "paths_path"
        ),
    )

    counts = np.bincount(
        np.asarray(
            labels,
            dtype=np.int64,
        ),
        minlength=2,
    )

    dataset_audit_rows.append(
        {
            "dataset": dataset_name,
            "n_total": int(
                len(
                    labels
                )
            ),
            "n_nonviolence": int(
                counts[
                    0
                ]
            ),
            "n_violence": int(
                counts[
                    1
                ]
            ),
            "cache_source": store[
                "cache_source"
            ],
            "color_order": store[
                "color_order"
            ],
            "feature_path": str(
                store[
                    "feature_path"
                ]
            ),
            "label_path": str(
                store[
                    "label_path"
                ]
            ),
        }
    )

dataset_audit_df = pd.DataFrame(
    dataset_audit_rows
)

dataset_audit_df.to_csv(
    OUTPUT_DIR
    / "heldout_dataset_audit.csv",
    index=False,
)

display(
    dataset_audit_df
)


DATASET: RWF
✅ RWF: legacy cache (BGR→RGB saat loading)

DATASET: UCF
✅ UCF: RGB-v2 cache

DATASET: BV
✅ BV: RGB-v2 cache


,dataset,n_total,n_nonviolence,n_violence,cache_source,color_order,feature_path,label_path
0,RWF,1998,1000,998,legacy,BGR,/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...
1,UCF,100,50,50,rgb64_v2,RGB,/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...
2,BV,1400,700,700,rgb64_v2,RGB,/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...


## 3. Definisi FrameMean persis seperti training

In [6]:
class EfficientFormerFrameMean(nn.Module):
    VALID_VARIANTS = {"frame_mean"}

    def __init__(
        self,
        variant: str = "frame_mean",
        backbone_name: str = "efficientformerv2_s0",
        num_classes: int = NUM_CLASSES,
        pretrained: bool = True,
    ):
        super().__init__()

        if variant not in self.VALID_VARIANTS:
            raise ValueError(f"Unknown variant: {variant}")

        self.variant = variant

        # Shared 2D backbone: setiap frame diproses independen
        # menggunakan bobot backbone yang sama.
        self.backbone = create_model(
            backbone_name,
            pretrained=pretrained,
            num_classes=0,
            in_chans=3,
        )
        self.feature_dim = int(self.backbone.num_features)

        # Dipertahankan agar arsitektur identik dengan training.
        self.feature_dropout = nn.Dropout(0.2)

        # TANPA BiLSTM dan TANPA temporal attention.
        self.classifier = nn.Sequential(
            nn.Linear(self.feature_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, num_classes),
        )
        self._initialize_classifier()

        # Explicit audit markers.
        self.lstm = None
        self.attention = None

    def _initialize_classifier(self):
        for module in self.classifier.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_uniform_(module.weight, nonlinearity="relu")
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, clip: torch.Tensor, return_attention: bool = False):
        # clip: [B,T,C,64,64], sudah RGB dan ImageNet-normalized.
        batch_size, seq_len, channels, height, width = clip.shape

        if channels != 3:
            raise ValueError(f"Expected 3 channels, got {channels}")

        # [B,T,C,H,W] -> [B*T,C,H,W]
        x = clip.reshape(
            batch_size * seq_len,
            channels,
            height,
            width,
        )

        x = F.interpolate(
            x,
            size=(BACKBONE_INPUT_SIZE, BACKBONE_INPUT_SIZE),
            mode="bilinear",
            align_corners=False,
        )

        features = self.backbone(x)

        if features.ndim == 4:
            features = F.adaptive_avg_pool2d(
                features,
                output_size=1,
            ).flatten(1)
        elif features.ndim != 2:
            features = features.flatten(1)

        # [B*T,D] -> [B,T,D]
        features = features.view(
            batch_size,
            seq_len,
            -1,
        )

        # Saat model.eval(), Dropout menjadi identity.
        features = self.feature_dropout(features)

        # FRAME MEAN: permutation-invariant terhadap dimensi temporal.
        # [B,T,D] -> [B,D]
        context = features.mean(dim=1)

        logits = self.classifier(context)

        if return_attention:
            return logits, None

        return logits


## 4. Muat checkpoint dengan `strict=True`

In [7]:
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint tidak ditemukan: {CHECKPOINT_PATH}"
    )

checkpoint = safe_torch_load(
    CHECKPOINT_PATH,
    map_location="cpu",
)

if not isinstance(
    checkpoint,
    dict,
):
    raise TypeError(
        "Checkpoint bukan dictionary."
    )

if "model_state_dict" not in checkpoint:
    raise KeyError(
        "Checkpoint tidak memiliki model_state_dict."
    )

checkpoint_config = checkpoint.get(
    "config",
    {},
)

if "variant" in checkpoint_config:
    if str(
        checkpoint_config[
            "variant"
        ]
    ) != VARIANT:
        raise RuntimeError(
            "Checkpoint variant mismatch."
        )

if "seed" in checkpoint_config:
    if int(
        checkpoint_config[
            "seed"
        ]
    ) != RUN_SEED:
        raise RuntimeError(
            "Checkpoint seed mismatch."
        )

if "sequence_length" in checkpoint_config:
    if int(
        checkpoint_config[
            "sequence_length"
        ]
    ) != SEQUENCE_LENGTH:
        raise RuntimeError(
            "Checkpoint sequence_length mismatch."
        )

model = EfficientFormerFrameMean(
    variant=VARIANT,
    pretrained=False,
).to(
    DEVICE
)

load_result = model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ],
    strict=True,
)

model.eval()

for parameter in model.parameters():
    parameter.requires_grad_(
        False
    )

if model.lstm is not None:
    raise RuntimeError(
        "FrameMean tidak boleh mempunyai LSTM."
    )

if model.attention is not None:
    raise RuntimeError(
        "FrameMean tidak boleh mempunyai attention."
    )

print(
    "✅ Strict FrameMean checkpoint/model audit PASS"
)

print(
    "Load result:",
    load_result,
)

✅ Strict FrameMean checkpoint/model audit PASS
Load result: <All keys matched successfully>


## 5. Dataset original dan permutasi

In [8]:
IMAGENET_MEAN = torch.tensor(
    [
        0.485,
        0.456,
        0.406,
    ],
    dtype=torch.float32,
).view(
    1,
    3,
    1,
    1,
)

IMAGENET_STD = torch.tensor(
    [
        0.229,
        0.224,
        0.225,
    ],
    dtype=torch.float32,
).view(
    1,
    3,
    1,
    1,
)


def stable_dataset_code(
    dataset_name,
):
    return int(
        zlib.crc32(
            dataset_name.encode(
                "utf-8"
            )
        )
        & 0xFFFFFFFF
    )


def make_permutation(
    dataset_name,
    source_idx,
    scenario,
    repeat_id,
):
    if scenario == "original":
        return np.arange(
            SEQUENCE_LENGTH,
            dtype=np.int64,
        )

    if scenario == "reverse":
        return np.arange(
            SEQUENCE_LENGTH - 1,
            -1,
            -1,
            dtype=np.int64,
        )

    if scenario == "shuffle":
        seed_sequence = np.random.SeedSequence(
            [
                int(
                    PERMUTATION_BASE_SEED
                ),
                int(
                    repeat_id
                ),
                int(
                    stable_dataset_code(
                        dataset_name
                    )
                ),
                int(
                    source_idx
                ),
            ]
        )

        rng = np.random.default_rng(
            seed_sequence
        )

        return rng.permutation(
            SEQUENCE_LENGTH
        ).astype(
            np.int64
        )

    raise ValueError(
        f"Unknown scenario: {scenario}"
    )


for scenario, repeat_id in [
    (
        "original",
        0,
    ),
    (
        "reverse",
        0,
    ),
    (
        "shuffle",
        0,
    ),
    (
        "shuffle",
        9,
    ),
]:
    a = make_permutation(
        "RWF",
        123,
        scenario,
        repeat_id,
    )

    b = make_permutation(
        "RWF",
        123,
        scenario,
        repeat_id,
    )

    if not np.array_equal(
        a,
        b,
    ):
        raise RuntimeError(
            "Permutation generator tidak deterministic."
        )

print(
    "✅ Permutation generator audit PASS"
)


class FrameMeanOrderDataset(
    Dataset
):
    def __init__(
        self,
        dataset_name,
        feature_path,
        label_path,
        color_order,
        paths_path,
        scenario,
        repeat_id=0,
    ):
        self.dataset_name = str(
            dataset_name
        )

        self.features = np.load(
            feature_path,
            mmap_mode="r",
        )

        self.labels = np.load(
            label_path,
            mmap_mode="r",
        ).astype(
            np.int64
        )

        self.color_order = str(
            color_order
        ).upper()

        self.paths_path = (
            Path(
                paths_path
            )
            if paths_path is not None
            else None
        )

        self.scenario = str(
            scenario
        )

        self.repeat_id = int(
            repeat_id
        )

    def __len__(
        self,
    ):
        return len(
            self.labels
        )

    def __getitem__(
        self,
        source_idx,
    ):
        frames = np.asarray(
            self.features[
                source_idx
            ],
            dtype=np.float32,
        )

        label = int(
            self.labels[
                source_idx
            ]
        )

        if not np.isfinite(
            frames
        ).all():
            raise ValueError(
                f"NaN/Inf pada {self.dataset_name}, idx={source_idx}"
            )

        frame_min = float(
            frames.min()
        )

        frame_max = float(
            frames.max()
        )

        if (
            AUTO_SCALE_LEGACY_CACHE
            and frame_min >= 0.0
            and frame_max > 1.5
        ):
            if frame_max <= 255.0 + 1e-3:
                frames = (
                    frames
                    / 255.0
                )
            else:
                raise ValueError(
                    f"Rentang cache tidak valid: "
                    f"[{frame_min},{frame_max}]"
                )

        if self.color_order == "BGR":
            frames = frames[
                ...,
                [
                    2,
                    1,
                    0,
                ],
            ]

        permutation = make_permutation(
            self.dataset_name,
            int(
                source_idx
            ),
            self.scenario,
            self.repeat_id,
        )

        selected = frames[
            permutation
        ]

        clip = torch.from_numpy(
            np.ascontiguousarray(
                selected
            )
        ).permute(
            0,
            3,
            1,
            2,
        )

        clip = (
            clip
            - IMAGENET_MEAN
        ) / IMAGENET_STD

        return (
            clip,
            torch.tensor(
                label,
                dtype=torch.long,
            ),
            int(
                source_idx
            ),
            torch.from_numpy(
                permutation.copy()
            ),
        )


def build_loader(
    dataset_name,
    scenario,
    repeat_id=0,
):
    (
        feature_path,
        label_path,
        color_order,
        paths_path,
    ) = TEST_STORES[
        dataset_name
    ]

    dataset = FrameMeanOrderDataset(
        dataset_name=dataset_name,
        feature_path=feature_path,
        label_path=label_path,
        color_order=color_order,
        paths_path=paths_path,
        scenario=scenario,
        repeat_id=repeat_id,
    )

    generator = torch.Generator()

    generator.manual_seed(
        PERMUTATION_BASE_SEED
    )

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=(
            DEVICE.type
            == "cuda"
        ),
        worker_init_fn=seed_worker,
        generator=generator,
        persistent_workers=(
            NUM_WORKERS
            > 0
        ),
    )


def load_paths_if_available(
    paths_path,
    expected_n,
):
    if paths_path is None:
        return None

    path = Path(
        paths_path
    )

    if not path.exists():
        return None

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as file:
        payload = json.load(
            file
        )

    paths = payload.get(
        "paths"
    )

    if not isinstance(
        paths,
        list,
    ):
        return None

    if len(
        paths
    ) != expected_n:
        return None

    return [
        str(
            path
        )
        for path in paths
    ]

✅ Permutation generator audit PASS


## 6. Evaluasi dan pembandingan logits

In [9]:
@torch.inference_mode()
def collect_outputs(
    dataset_name,
    scenario,
    repeat_id,
):
    loader = build_loader(
        dataset_name,
        scenario,
        repeat_id,
    )

    model.eval()

    all_indices = []
    all_labels = []
    all_logits = []
    all_probabilities = []
    all_permutations = []

    for (
        clips,
        labels,
        indices,
        permutations,
    ) in tqdm(
        loader,
        leave=False,
        desc=f"{dataset_name} | {scenario} | {repeat_id}",
    ):
        clips = clips.to(
            DEVICE,
            non_blocking=True,
        )

        # Deliberately no AMP/autocast.
        logits = model(
            clips
        ).float()

        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        all_indices.extend(
            np.asarray(
                indices
            ).tolist()
        )

        all_labels.extend(
            labels.numpy().tolist()
        )

        all_logits.extend(
            logits.cpu().numpy().tolist()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy().tolist()
        )

        all_permutations.extend(
            permutations.numpy().tolist()
        )

    indices_np = np.asarray(
        all_indices,
        dtype=np.int64,
    )

    labels_np = np.asarray(
        all_labels,
        dtype=np.int64,
    )

    logits_np = np.asarray(
        all_logits,
        dtype=np.float64,
    )

    probabilities_np = np.asarray(
        all_probabilities,
        dtype=np.float64,
    )

    predictions_np = probabilities_np.argmax(
        axis=1
    )

    (
        feature_path,
        label_path,
        color_order,
        paths_path,
    ) = TEST_STORES[
        dataset_name
    ]

    source_paths = load_paths_if_available(
        paths_path,
        expected_n=len(
            indices_np
        ),
    )

    if source_paths is None:
        sample_keys = [
            f"{dataset_name}::{int(index)}"
            for index in indices_np
        ]

        video_paths = [
            ""
            for _ in indices_np
        ]

    else:
        video_paths = [
            source_paths[
                int(
                    index
                )
            ]
            for index in indices_np
        ]

        sample_keys = [
            Path(
                path
            ).as_posix()
            for path in video_paths
        ]

    condition_predictions_df = pd.DataFrame(
        {
            "checkpoint_seed": RUN_SEED,
            "dataset": dataset_name,
            "scenario": scenario,
            "repeat": repeat_id,
            "sample_index": indices_np,
            "sample_key": sample_keys,
            "video_path": video_paths,
            "permutation_0based": [
                json.dumps(
                    permutation
                )
                for permutation in all_permutations
            ],
            "y_true": labels_np,
            "y_pred": predictions_np,
            "is_correct": (
                labels_np
                == predictions_np
            ).astype(
                np.int64
            ),
            "logit_nonviolence": logits_np[
                :,
                0
            ],
            "logit_violence": logits_np[
                :,
                1
            ],
            "p_nonviolence": probabilities_np[
                :,
                0
            ],
            "p_violence": probabilities_np[
                :,
                1
            ],
        }
    )

    if condition_predictions_df[
        "sample_key"
    ].duplicated().any():
        raise RuntimeError(
            f"Duplicate sample_key: "
            f"{dataset_name}, {scenario}, repeat={repeat_id}."
        )

    return {
        "indices": indices_np,
        "labels": labels_np,
        "predictions": predictions_np,
        "logits": logits_np,
        "probabilities": probabilities_np,
        "predictions_df": condition_predictions_df,
    }


def compute_metrics(
    labels,
    probabilities,
):
    predictions = probabilities.argmax(
        axis=1
    )

    (
        precision_cls,
        recall_cls,
        f1_cls,
        support,
    ) = precision_recall_fscore_support(
        labels,
        predictions,
        labels=[
            0,
            1,
        ],
        zero_division=0,
    )

    metrics = {
        "accuracy": float(
            accuracy_score(
                labels,
                predictions,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                labels,
                predictions,
            )
        ),
        "precision_macro": float(
            precision_score(
                labels,
                predictions,
                average="macro",
                zero_division=0,
            )
        ),
        "recall_macro": float(
            recall_score(
                labels,
                predictions,
                average="macro",
                zero_division=0,
            )
        ),
        "f1_macro": float(
            f1_score(
                labels,
                predictions,
                average="macro",
                zero_division=0,
            )
        ),
        "precision_nonviolence": float(
            precision_cls[
                0
            ]
        ),
        "recall_nonviolence": float(
            recall_cls[
                0
            ]
        ),
        "f1_nonviolence": float(
            f1_cls[
                0
            ]
        ),
        "precision_violence": float(
            precision_cls[
                1
            ]
        ),
        "recall_violence": float(
            recall_cls[
                1
            ]
        ),
        "f1_violence": float(
            f1_cls[
                1
            ]
        ),
    }

    if len(
        np.unique(
            labels
        )
    ) == 2:
        metrics[
            "roc_auc"
        ] = float(
            roc_auc_score(
                labels,
                probabilities[
                    :,
                    1
                ],
            )
        )

        metrics[
            "average_precision"
        ] = float(
            average_precision_score(
                labels,
                probabilities[
                    :,
                    1
                ],
            )
        )

    else:
        metrics[
            "roc_auc"
        ] = np.nan

        metrics[
            "average_precision"
        ] = np.nan

    return metrics


def center_logits(
    logits,
):
    return (
        logits
        - logits.mean(
            axis=1,
            keepdims=True,
        )
    )


def compare_outputs(
    original,
    perturbed,
):
    if not np.array_equal(
        original[
            "indices"
        ],
        perturbed[
            "indices"
        ],
    ):
        raise RuntimeError(
            "Urutan sample berubah."
        )

    if not np.array_equal(
        original[
            "labels"
        ],
        perturbed[
            "labels"
        ],
    ):
        raise RuntimeError(
            "Label berubah."
        )

    original_metrics = compute_metrics(
        original[
            "labels"
        ],
        original[
            "probabilities"
        ],
    )

    perturbed_metrics = compute_metrics(
        perturbed[
            "labels"
        ],
        perturbed[
            "probabilities"
        ],
    )

    raw_logit_diff = np.abs(
        original[
            "logits"
        ]
        - perturbed[
            "logits"
        ]
    )

    centered_logit_diff = np.abs(
        center_logits(
            original[
                "logits"
            ]
        )
        - center_logits(
            perturbed[
                "logits"
            ]
        )
    )

    probability_diff = np.abs(
        original[
            "probabilities"
        ]
        - perturbed[
            "probabilities"
        ]
    )

    original_margin = (
        original[
            "logits"
        ][
            :,
            1
        ]
        - original[
            "logits"
        ][
            :,
            0
        ]
    )

    perturbed_margin = (
        perturbed[
            "logits"
        ][
            :,
            1
        ]
        - perturbed[
            "logits"
        ][
            :,
            0
        ]
    )

    result = {
        "max_abs_raw_logit_diff": float(
            raw_logit_diff.max()
        ),
        "max_abs_centered_logit_diff": float(
            centered_logit_diff.max()
        ),
        "max_abs_logit_margin_diff": float(
            np.max(
                np.abs(
                    original_margin
                    - perturbed_margin
                )
            )
        ),
        "max_abs_probability_diff": float(
            probability_diff.max()
        ),
        "changed_predictions": int(
            np.sum(
                original[
                    "predictions"
                ]
                != perturbed[
                    "predictions"
                ]
            )
        ),
    }

    for metric_name in [
        "accuracy",
        "balanced_accuracy",
        "f1_macro",
        "roc_auc",
        "average_precision",
    ]:
        result[
            f"{metric_name}_original"
        ] = original_metrics[
            metric_name
        ]

        result[
            f"{metric_name}_perturbed"
        ] = perturbed_metrics[
            metric_name
        ]

        result[
            f"delta_{metric_name}"
        ] = (
            original_metrics[
                metric_name
            ]
            - perturbed_metrics[
                metric_name
            ]
        )

    return result

## 7. Jalankan sanity check pada seluruh dataset

In [10]:
comparison_rows = []

metric_rows = []

prediction_frames = []

SCENARIOS = [
    (
        "reverse",
        0,
    ),
] + [
    (
        "shuffle",
        repeat_id,
    )
    for repeat_id in range(
        NUM_RANDOM_SHUFFLES
    )
]

for dataset_name in TEST_STORES.keys():
    print(
        "\n"
        + "=" * 90
    )

    print(
        "Dataset:",
        dataset_name,
    )

    print(
        "=" * 90
    )

    original = collect_outputs(
        dataset_name,
        "original",
        0,
    )

    original_metrics = compute_metrics(
        original[
            "labels"
        ],
        original[
            "probabilities"
        ],
    )

    metric_rows.append(
        {
            "checkpoint_seed": RUN_SEED,
            "dataset": dataset_name,
            "scenario": "original",
            "repeat": 0,
            **original_metrics,
        }
    )

    prediction_frames.append(
        original[
            "predictions_df"
        ]
    )

    for (
        scenario,
        repeat_id,
    ) in SCENARIOS:
        perturbed = collect_outputs(
            dataset_name,
            scenario,
            repeat_id,
        )

        perturbed_metrics = compute_metrics(
            perturbed[
                "labels"
            ],
            perturbed[
                "probabilities"
            ],
        )

        metric_rows.append(
            {
                "checkpoint_seed": RUN_SEED,
                "dataset": dataset_name,
                "scenario": scenario,
                "repeat": repeat_id,
                **perturbed_metrics,
            }
        )

        prediction_frames.append(
            perturbed[
                "predictions_df"
            ]
        )

        comparison_rows.append(
            {
                "checkpoint_seed": RUN_SEED,
                "dataset": dataset_name,
                "scenario": scenario,
                "repeat": repeat_id,
                **compare_outputs(
                    original,
                    perturbed,
                ),
            }
        )

comparison_df = pd.DataFrame(
    comparison_rows
)

metrics_df = pd.DataFrame(
    metric_rows
)

all_predictions_df = pd.concat(
    prediction_frames,
    ignore_index=True,
)

comparison_path = (
    OUTPUT_DIR
    / f"framemean_temporal_order_comparison_seed{RUN_SEED}.csv"
)

metrics_path = (
    OUTPUT_DIR
    / f"framemean_temporal_order_metrics_seed{RUN_SEED}.csv"
)

predictions_path = (
    OUTPUT_DIR
    / f"framemean_temporal_order_predictions_seed{RUN_SEED}.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False,
)

metrics_df.to_csv(
    metrics_path,
    index=False,
)

all_predictions_df.to_csv(
    predictions_path,
    index=False,
)

print(
    "\nSaved:",
    comparison_path,
)

print(
    "Saved:",
    metrics_path,
)

print(
    "Saved:",
    predictions_path,
)

display(
    comparison_df.round(
        10
    )
)


Dataset: RWF


RWF | original | 0:   0%|          | 0/250 [00:00<?, ?it/s]

RWF | reverse | 0:   0%|          | 0/250 [00:00<?, ?it/s]

RWF | shuffle | 0:   0%|          | 0/250 [00:00<?, ?it/s]

RWF | shuffle | 1:   0%|          | 0/250 [00:00<?, ?it/s]

...

BV | shuffle | 7:   0%|          | 0/175 [00:00<?, ?it/s]

BV | shuffle | 8:   0%|          | 0/175 [00:00<?, ?it/s]

BV | shuffle | 9:   0%|          | 0/175 [00:00<?, ?it/s]


Saved: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/FrameMean_TemporalOrder/seed_0/framemean_temporal_order_comparison_seed0.csv
Saved: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/FrameMean_TemporalOrder/seed_0/framemean_temporal_order_metrics_seed0.csv
Saved: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/FrameMean_TemporalOrder/seed_0/framemean_temporal_order_predictions_seed0.csv


,checkpoint_seed,dataset,scenario,repeat,max_abs_raw_logit_diff,max_abs_centered_logit_diff,max_abs_logit_margin_diff,max_abs_probability_diff,changed_predictions,accuracy_original,...,delta_balanced_accuracy,f1_macro_original,f1_macro_perturbed,delta_f1_macro,roc_auc_original,roc_auc_perturbed,delta_roc_auc,average_precision_original,average_precision_perturbed,delta_average_precision
0,0,RWF,reverse,0,9.765625e-04,5.874634e-04,0.001175,4.768000e-07,0,0.598599,...,0.0,0.594370,0.594370,0.0,0.630863,0.630863,0.0,0.591761,0.591761,0.000000e+00
1,0,RWF,shuffle,0,9.765625e-04,7.324219e-04,0.001465,4.172000e-07,0,0.598599,...,0.0,0.594370,0.594370,0.0,0.630863,0.630863,0.0,0.591761,0.591761,2.910000e-07
2,0,RWF,shuffle,1,5.340576e-04,5.264282e-04,0.001053,4.172000e-07,0,0.598599,...,0.0,0.594370,0.594370,0.0,0.630863,0.630863,0.0,0.591761,0.591761,2.910000e-07
3,0,RWF,shuffle,2,2.593994e-04,1.983643e-04,0.000397,4.470000e-07,0,0.598599,...,0.0,0.594370,0.594370,0.0,0.630863,0.630863,0.0,0.591761,0.591760,5.485000e-07
4,0,RWF,shuffle,3,5.798340e-04,5.416870e-04,0.001083,4.768000e-07,0,0.598599,...,0.0,0.594370,0.594370,0.0,0.630863,0.630863,0.0,0.591761,0.591760,1.122000e-06
5,0,RWF,shuffle,4,5.187988e-04,4.425049e-04,0.000885,4.768000e-07,0,0.598599,...,0.0,0.594370,0.594370,0.0,0.630863,0.630863,0.0,0.591761,0.591760,1.122000e-06
6,0,RWF,shuffle,5,4.882812e-04,4.501343e-04,0.000900,4.172000e-07,0,0.598599,...,0.0,0.594370,0.594370,0.0,0.630863,0.630863,0.0,0.591761,0.591761,2.910000e-07
7,0,RWF,shuffle,6,5.187988e-04,4.577637e-04,0.000916,4.172000e-07,0,0.598599,...,0.0,0.594370,0.594370,0.0,0.630863,0.630863,0.0,0.591761,0.591760,8.395000e-07
8,0,RWF,shuffle,7,3.967285e-04,3.814697e-04,0.000763,4.172000e-07,0,0.598599,...,0.0,0.594370,0.594370,0.0,0.630863,0.630863,0.0,0.591761,0.591760,5.735000e-07
9,0,RWF,shuffle,8,7.629395e-04,6.561279e-04,0.001312,4.768000e-07,0,0.598599,...,0.0,0.594370,0.594370,0.0,0.630863,0.630863,0.0,0.591761,0.591760,8.395000e-07


## 8. Pooled check dan gerbang validitas akhir


In [11]:
summary_rows = []

for dataset_name, group in comparison_df.groupby(
    "dataset"
):
    summary_rows.append(
        {
            "checkpoint_seed": RUN_SEED,
            "dataset": dataset_name,
            "num_perturbations": int(
                len(
                    group
                )
            ),
            "max_abs_raw_logit_diff": float(
                group[
                    "max_abs_raw_logit_diff"
                ].max()
            ),
            "max_abs_centered_logit_diff": float(
                group[
                    "max_abs_centered_logit_diff"
                ].max()
            ),
            "max_abs_logit_margin_diff": float(
                group[
                    "max_abs_logit_margin_diff"
                ].max()
            ),
            "max_abs_probability_diff": float(
                group[
                    "max_abs_probability_diff"
                ].max()
            ),
            "total_changed_predictions": int(
                group[
                    "changed_predictions"
                ].sum()
            ),
            "max_abs_accuracy_delta": float(
                group[
                    "delta_accuracy"
                ].abs().max()
            ),
            "max_abs_balanced_accuracy_delta": float(
                group[
                    "delta_balanced_accuracy"
                ].abs().max()
            ),
            "max_abs_f1_macro_delta": float(
                group[
                    "delta_f1_macro"
                ].abs().max()
            ),
            "max_abs_roc_auc_delta": float(
                group[
                    "delta_roc_auc"
                ].abs().max()
            ),
            "max_abs_average_precision_delta": float(
                group[
                    "delta_average_precision"
                ].abs().max()
            ),
        }
    )

summary_df = pd.DataFrame(
    summary_rows
)

summary_path = (
    OUTPUT_DIR
    / f"framemean_temporal_order_summary_seed{RUN_SEED}.csv"
)

summary_df.to_csv(
    summary_path,
    index=False,
)

display(
    summary_df
)

print(
    "Saved:",
    summary_path,
)

# ============================================================
# HARD INVARIANCE GATE — PER DATASET
# ============================================================
for _, row in summary_df.iterrows():
    dataset_name = str(
        row[
            "dataset"
        ]
    )

    max_prob_diff = float(
        row[
            "max_abs_probability_diff"
        ]
    )

    if not np.isfinite(
        max_prob_diff
    ):
        raise RuntimeError(
            f"{dataset_name}: probability difference NaN/Inf."
        )

    if max_prob_diff > PROB_ATOL:
        raise RuntimeError(
            f"{dataset_name}: probability difference "
            f"{max_prob_diff:.12g} > {PROB_ATOL:.12g}."
        )

    if int(
        row[
            "total_changed_predictions"
        ]
    ) != 0:
        raise RuntimeError(
            f"{dataset_name}: ada class prediction yang berubah."
        )

    for metric_name in [
        "max_abs_accuracy_delta",
        "max_abs_balanced_accuracy_delta",
        "max_abs_f1_macro_delta",
    ]:
        if float(
            row[
                metric_name
            ]
        ) > METRIC_ATOL:
            raise RuntimeError(
                f"{dataset_name}: {metric_name} melebihi toleransi."
            )

    if float(
        row[
            "max_abs_roc_auc_delta"
        ]
    ) > AUC_ATOL:
        raise RuntimeError(
            f"{dataset_name}: ROC-AUC berubah melebihi toleransi."
        )

    if float(
        row[
            "max_abs_average_precision_delta"
        ]
    ) > AP_ATOL:
        raise RuntimeError(
            f"{dataset_name}: AP berubah melebihi toleransi."
        )

protocol_audit = {
    "task": "heldout_temporal_order_evaluation_only",
    "model": "EffTempNet FrameMean",
    "variant": VARIANT,
    "checkpoint_seed": RUN_SEED,
    "checkpoint_path": str(
        CHECKPOINT_PATH
    ),
    "training_performed": False,
    "fine_tuning_performed": False,
    "strict_checkpoint_loading": True,
    "sequence_length": SEQUENCE_LENGTH,
    "aggregation": "mean_of_per_frame_features",
    "scenarios": [
        "original",
        "reverse",
        "shuffle_x10",
    ],
    "num_random_shuffles": NUM_RANDOM_SHUFFLES,
    "permutation_base_seed": PERMUTATION_BASE_SEED,
    "checkpoint_seed_excluded_from_permutation_generation": True,
    "use_amp": False,
    "heldout_datasets": list(
        HELDOUT_SPECS.keys()
    ),
    "no_cross_dataset_pooling": True,
    "probability_tolerance": PROB_ATOL,
    "metric_tolerance": METRIC_ATOL,
    "auc_tolerance": AUC_ATOL,
    "ap_tolerance": AP_ATOL,
}

manifest_path = (
    OUTPUT_DIR
    / f"framemean_temporal_order_protocol_seed{RUN_SEED}.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        protocol_audit,
        file,
        indent=2,
    )

print(
    "\n✅ FrameMean held-out temporal-order invariance PASS "
    "untuk RWF, UCF, dan Bus Violence."
)

print(
    "Protocol:",
    manifest_path,
)

,checkpoint_seed,dataset,num_perturbations,max_abs_raw_logit_diff,max_abs_centered_logit_diff,max_abs_logit_margin_diff,max_abs_probability_diff,total_changed_predictions,max_abs_accuracy_delta,max_abs_balanced_accuracy_delta,max_abs_f1_macro_delta,max_abs_roc_auc_delta,max_abs_average_precision_delta
0,0,BV,11,0.000002,1.445413e-06,0.000003,7.152557e-07,0,0.0,0.0,0.0,0.000000e+00,0.000000
1,0,RWF,11,0.000977,7.324219e-04,0.001465,4.768372e-07,0,0.0,0.0,0.0,2.220446e-16,0.000001
2,0,UCF,11,0.000001,8.344650e-07,0.000002,3.576279e-07,0,0.0,0.0,0.0,0.000000e+00,0.000000


Saved: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/FrameMean_TemporalOrder/seed_0/framemean_temporal_order_summary_seed0.csv


RuntimeError: RWF: AP berubah melebihi toleransi.

## Seed berikutnya

Cukup ubah:

```python
RUN_SEED = 0
```

menjadi:

```python
RUN_SEED = 42
```

kemudian:

```python
RUN_SEED = 123
```

Jangan ubah:

```python
SEQUENCE_LENGTH = 16
NUM_RANDOM_SHUFFLES = 10
PERMUTATION_BASE_SEED = 20260810
BATCH_SIZE = 8
PROB_ATOL = 1e-6
```

Output:

```text
HeldOut_Evaluation/
└── FrameMean_TemporalOrder/
    ├── seed_0/
    ├── seed_42/
    └── seed_123/
```

File utama:
- `framemean_temporal_order_comparison_seedX.csv`
- `framemean_temporal_order_metrics_seedX.csv`
- `framemean_temporal_order_predictions_seedX.csv`
- `framemean_temporal_order_summary_seedX.csv`
- `framemean_temporal_order_protocol_seedX.json`
- `heldout_dataset_audit.csv`